In [ ]:
# Cell 1: Shared state + sample input shape
from typing import TypedDict, List, Dict, Any
from pprint import pprint


class WorkflowState(TypedDict):
    input: Dict[str, Any]
    pending_specialists: List[str]
    specialist_outputs: Dict[str, Any]
    final_output: Dict[str, Any]

In [ ]:
# Cell 2: Orchestrator decides who should handle the request
def orchestrator(state: WorkflowState) -> WorkflowState:
    data = state["input"]
    specialists = []

    # Minimal routing rules based on input data
    if data.get("category") in {"billing", "refund"}:
        specialists.append("billing_specialist")
    if data.get("category") in {"tech", "bug", "incident"}:
        specialists.append("tech_specialist")
    if data.get("priority") == "high":
        specialists.append("escalation_specialist")

    # Fallback if no rule matched
    if not specialists:
        specialists.append("general_specialist")

    state["pending_specialists"] = specialists
    return state

In [ ]:
# Cell 3: Specialist agents (simple placeholders)
def billing_specialist(data: Dict[str, Any]) -> Dict[str, Any]:
    return {"action": "check_invoice_and_refund_policy", "owner": "billing_team"}


def tech_specialist(data: Dict[str, Any]) -> Dict[str, Any]:
    return {"action": "diagnose_issue_and_collect_logs", "owner": "engineering_support"}


def escalation_specialist(data: Dict[str, Any]) -> Dict[str, Any]:
    return {"action": "page_oncall_and_raise_priority", "owner": "incident_manager"}


def general_specialist(data: Dict[str, Any]) -> Dict[str, Any]:
    return {"action": "triage_and_request_more_context", "owner": "support_triage"}


SPECIALIST_MAP = {
    "billing_specialist": billing_specialist,
    "tech_specialist": tech_specialist,
    "escalation_specialist": escalation_specialist,
    "general_specialist": general_specialist,
}

In [ ]:
# Cell 4: Runner (handover loop)
def run_workflow(input_data: Dict[str, Any]) -> WorkflowState:
    state: WorkflowState = {
        "input": input_data,
        "pending_specialists": [],
        "specialist_outputs": {},
        "final_output": {},
    }

    # Step 1: Orchestrator picks one or more specialists
    state = orchestrator(state)

    # Step 2: Handover to each selected specialist
    while state["pending_specialists"]:
        specialist_name = state["pending_specialists"].pop(0)
        fn = SPECIALIST_MAP[specialist_name]
        result = fn(state["input"])
        state["specialist_outputs"][specialist_name] = result

    # Step 3: Merge outputs (simple aggregator)
    state["final_output"] = {
        "request_id": state["input"].get("request_id"),
        "handled_by": list(state["specialist_outputs"].keys()),
        "next_actions": [v["action"] for v in state["specialist_outputs"].values()],
    }

    return state

In [ ]:
# Cell 5: Try a few inputs
examples = [
    {"request_id": "R-101", "category": "refund", "priority": "low"},
    {"request_id": "R-102", "category": "incident", "priority": "high"},
    {"request_id": "R-103", "category": "unknown", "priority": "low"},
]

for ex in examples:
    print("\nINPUT:", ex)
    out = run_workflow(ex)
    pprint(out["final_output"])